# Session 05: Generated Text as Data

**Student implementer and maintainer:** YQ-HA. This notebook records code typed,
executed, and adapted while following the instructor during class, together with
the student's post-course completions and maintenance.

**Course:** Tyler Shoemaker, [Introduction to Interpretability for Language Models](https://digitaltheorylab.github.io/2026_interpretability-reader/).
Reference implementation: [05_text.md](https://github.com/digitaltheorylab/2026_interpretability-reader/blob/main/chapters/05_text.md).

**Repair assistance:** OpenAI Codex assisted with locating and explaining errors,
restoring omitted course code, and checking coverage and runnability. `Repair:`
comments identify corrected mistakes. Original classroom notes, including `##`
comments, are retained.

Run from the first cell with the kernel in `notebooks/`. New results go to
`results/session-05/`. This repaired version completed its Colab execution on October 26, 2026. See `results/` for the saved run; session 08 analyzed a truncated reasoning trace.


In [ ]:
from pathlib import Path
import json
import platform
import sys
import matplotlib.pyplot as plt
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results" / "session-05"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FACET_HEIGHT = 4
FACET_ASPECT = 1.2
def save_figure(name):
    plt.gcf().savefig(RESULTS_DIR / name, bbox_inches="tight", dpi=150)
    plt.show()
    plt.close("all")


In [ ]:
from string import Template

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.optimize import minimize
from scipy.special import logsumexp, rel_entr
from scipy.stats import entropy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import MultinomialNB

In [ ]:
SYSTEM_PROMPT = (
    "You are a creative-writing assistant. "
    "Write only the requested story text. Never add commentary."
)

PROMPT = Template(
    "Write a complete$genre short story of no more than 1,000 words. "
    "The story must have a clear ending. "
    "Do not include a title, preface, notes, or commentary.\n\nStory:\n"
)

GENRES = {
    "Horror": "horror",
    "Detective fiction": "detective fiction",
    "Romance": "romance",
    "Science fiction": "science fiction",
    "Thriller": "thriller",
}

genre = GENRES["Horror"]
prompt = PROMPT.substitute(genre=f" {genre}" if genre else "")
print(prompt)



In [ ]:
genre = None
prompt = PROMPT.substitute(genre=f" {genre}" if genre else "")
print(prompt)

In [ ]:
# load data set
df = pd.read_parquet(DATA_DIR / "w2-d1_genre-stories.parquet")


In [ ]:
df # tabluar data: ~2D matrix (rows, cols) --> spreadsheet

In [ ]:
print(df["text"].sample().item())

In [ ]:
df["genre"].value_counts()

In [ ]:
df.columns

In [ ]:
df["text"].sample().item()

In [ ]:
print(df["text"].sample().item())

In [ ]:
# document -term matrix: 2D matrix of term counts over documents.
#
# \ d/t \ t1 \ t2 \ t3 \
# \ d1 \ 0 \ 1 \ 0 > G2
# \ d2 \ 3 \ 0 \ 1 \ > G1*
# \ d3 \ 3 \ 0 \ 1 \ > G1*

# scikit-learn's CountVectorizer produces a document-term matrix
vectorizer = CountVectorizer(
    token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
    stop_words="english", # common words like 'the', 'a', 'and', but also 'time', 'he', 'she'"
    max_df = 0.9, # Max doc frequency: if a word appears in 90% or more of outputs: ignore
    min_df=10, # Min document frequency: if a word appears in 10 or less outputs: ignore.
)


# scikit-learn's object: fit, transform (use what you learned to make a data representation), predict (use a fitted object to make prediction)
#
#
# model.fit()
# model.fit_predict()

X = vectorizer.fit_transform(df["text"].tolist())

feature_names = vectorizer.get_feature_names_out()
y = df["genre"].to_numpy()



In [ ]:
X

In [ ]:
y

In [ ]:
# Repair: Display a small slice while retaining the full sparse X for fitting.
pd.DataFrame(X[:10, :20].toarray(), columns=feature_names[:20]) # Document-term matrix

In [ ]:
df.shape[0] == y.shape[0]

In [ ]:
# Repair: fit is a method of the classifier instance, not a standalone function.
# What is the probability of token t appearing in genre g?
#
# fit a classifier that tries to estimate G_{0-5} from T_{i}, where i = document of token counts
#
# Learn a probability distribution of token counts over genres
# Then, for a given document, sum the log probability distribution of its token counts and estimate its probability vis-a-vis all genres
# Assign a genre label to the genre with the highest probability (argmax)
#
# Multinomial: ~proportional data over categories
# - Text daata  loves this kind of thing because tokens are categories
# - We used multinomial sampling over model vocabulary
# - Certain genres will share words: detective fiction and thriller --> "gun" --> what is the p(g) in DF and T?
# --> what is the p(gunn) in DF and T? What is the p(genre_DF or genre_T) given p(gun) + p (other tokens)?

clf = MultinomialNB()
clf.fit(X, y)


In [ ]:
np.exp(clf.class_log_prior_) # convert log probs to probs with e ** x

In [ ]:
# Repair: Correct DataFrame capitalization and use parentheses for predict_proba.
# given our first document's term counts, what's the probability distribution over genre?
# genre is also multinomial, i.e., a set of categories

pd.DataFrame(clf.predict_proba(X[0]), columns=clf.classes_) # underscores are another annoying convention scikit-learn.

In [ ]:
acc = clf.score(X, y)
print("Mean accuracy:", round(acc, 3))


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold

# Fit the vocabulary inside each fold; the full-data fit above is for word analysis.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=5167)
evaluation_model = make_pipeline(
    CountVectorizer(token_pattern=r"(?u)\b[a-zA-Z]{3,}\b", stop_words="english", max_df=0.9, min_df=10),
    MultinomialNB(),
)
cv_scores = cross_val_score(evaluation_model, df["text"], y, cv=cv, scoring="accuracy")
print("Five-fold accuracy:", cv_scores.mean(), "Fold scores:", cv_scores)


In [ ]:
baseline = df["genre"].value_counts(normalize=True).max()
if cv_scores.mean() < baseline + 0.05:
    print("WARNING: accuracy ~ baseline. Genres are likely not separable")
else:
    print("OK: model can separate genres")


In [ ]:
classes = clf.classes_
classes


In [ ]:
log_p_t_given_g = clf.feature_log_prob_
log_p_g = clf.class_log_prior_


In [ ]:
np.exp(log_p_g)


In [ ]:
log_p_t = logsumexp(log_p_g[:, None] + log_p_t_given_g, axis=0)


In [ ]:
pmi = log_p_t_given_g - log_p_t


In [ ]:
log_p_joint = log_p_g[:, None] + log_p_t_given_g
p_joint = np.exp(log_p_joint)
wpmi = p_joint * pmi


In [ ]:
pmi_df  = pd.DataFrame(pmi,  index=classes, columns=feature_names)
wpmi_df = pd.DataFrame(wpmi, index=classes, columns=feature_names)


In [ ]:
print("Shape of DataFrame:", pmi_df.shape)


In [ ]:
def top_tokens(wpmi_df, pmi_df, genre, k=25):
    """Return the ``k`` most distinctive tokens for a genre.

    Parameters
    ----------
    wpmi_df : pd.DataFrame
        Weighted PMI scores, shape (n_genre, vocab_size)
    pmi_df : pd.DataFrame
        PMI scores, shape (n_genre, vocab_size)
    genre : str
        Genre to rank tokens for
    k : int
        Number of tokens to return

    Returns
    -------
    pd.DataFrame
        Top ``k`` tokens for ``genre``, with their wPMI and lift scores
    """
    # Get the k tokens with the highest wPMI for this genre
    top = wpmi_df.loc[genre].nlargest(k)

    # Calculate lift for these tokens
    lift = np.exp(pmi_df.loc[genre, top.index])

    # Package up into a DataFrame
    df = pd.DataFrame({"wPMI": top, "lift": lift}).reset_index(names="token")

    return df


In [ ]:
results = []
for genre in classes:
    top = top_tokens(wpmi_df, pmi_df, genre)
    top["genre"] = genre
    results.append(top)


In [ ]:
top_plot = pd.concat(results, ignore_index=True)


In [ ]:
g = sns.catplot(
    data=top_plot,
    x="wPMI",
    y="token",
    col="genre",
    col_wrap=2,
    kind="bar",
    sharey=False,
    height=FACET_HEIGHT,
    aspect=FACET_ASPECT,
)
g.set_axis_labels("weighted PMI", "")
g.set_titles("{col_name}")

for genre, ax in g.axes_dict.items():
    subset = top_plot[top_plot["genre"] == genre]

    for patch, (_, row) in zip(ax.patches, subset.iterrows()):
        x_pos = patch.get_width()
        y_pos = patch.get_y() + patch.get_height() / 2

        ax.text(
            x_pos,
            y_pos,
            f"  {row['lift']:.1f}x",
            va="center",
            ha="left",
        )

    ax.set_xlim(right=ax.get_xlim()[1] * 1.15)

g.fig.suptitle(
    "Distinctive Tokens by Genre: wPMI with Lift Labels", y=1.05
)
save_figure(RESULTS_DIR / "figure-02.png")


In [ ]:
probs = np.exp(log_p_t_given_g)
P = pd.DataFrame(probs, index=classes, columns=feature_names)


In [ ]:
explicit_genres = []
for genre in classes:
    if genre == "NULL":
        continue

    explicit_genres.append(genre)

Q = P.loc[explicit_genres].to_numpy()
p_null = P.loc["NULL"].to_numpy()


In [ ]:
kld = np.apply_along_axis(lambda q: entropy(P.loc["NULL"], q), axis=1, arr=Q)


In [ ]:
kld_plot = pd.DataFrame({"KLD": kld, "genre": explicit_genres})
kld_plot.sort_values("KLD", ascending=True, inplace=True)


In [ ]:
fig, ax = plt.subplots()
sns.barplot(data=kld_plot, x="KLD", y="genre", ax=ax)
ax.set(title="KL Divergence", xlabel="KL divergence (nats)", ylabel="Genre")

for patch, label in zip(ax.containers[0].patches, kld_plot["KLD"]):
    x_pos = patch.get_width()
    y_pos = patch.get_y() + patch.get_height() / 2
    ax.text(
        x_pos,
        y_pos,
        f"  {label:.3f}",
        va="center",
        ha="left",
     )

ax.set_xlim(right=ax.get_xlim()[1] * 1.15)
save_figure(RESULTS_DIR / "figure-03.png")


In [ ]:
prob_plot = (
    P
    .rename_axis(index="genre", columns="token")
    .stack()
    .rename("probability")
    .reset_index()
)


In [ ]:
fig, ax = plt.subplots()
sns.kdeplot(
    data=prob_plot,
    x="probability",
    hue="genre",
    multiple="layer",
    log_scale=True,
    ax=ax,
)
ax.set(title="Token Log Probabilities Per Genre", xlabel="Log probability")
save_figure(RESULTS_DIR / "figure-04.png")


In [ ]:
def mixture_kl(w):
    """Return KL divergence between NULL and a weighted genre mixture.

    Parameters
    ----------
    w : np.ndarray
        Mixture weights for the explicit genre distributions, shape (n_genre,)

    Returns
    -------
    float
        The KL divergence
    """
    # Construct the mixture distribution: q[t] = sum_g w[g] * Q[g, t]
    q = w @ Q

    # Avoid log(0) if any token receives zero probability and re-normalize so q
    # remains a probability distribution
    q = np.clip(q, 1e-300, None)
    q = q / q.sum()

    return entropy(p_null, q)


In [ ]:
x0 = np.ones(len(explicit_genres)) / len(explicit_genres)
constraints = {"type": "eq", "fun": lambda w: w.sum() - 1}
bounds = [(0, 1)] * len(explicit_genres)


In [ ]:
res = minimize(
    mixture_kl, x0, method="SLSQP", bounds=bounds, constraints=constraints
)

if not res.success:
    raise RuntimeError(res.message)
else:
    print("OK: optimizer found a solution")


In [ ]:
weighting = res.x / res.x.sum()
mix_plot = pd.DataFrame({"mixture": weighting, "genre": explicit_genres})
mix_plot.sort_values("mixture", ascending=False, inplace=True)


In [ ]:
fig, ax = plt.subplots()
sns.barplot(data=mix_plot, x="mixture", y="genre", ax=ax)
ax.set(title="Genre mixture for NULL", xlabel="Mixture weight", ylabel="Genre")

for patch, label in zip(ax.containers[0].patches, mix_plot["mixture"]):
    x_pos = patch.get_width()
    y_pos = patch.get_y() + patch.get_height() / 2
    ax.text(
        x_pos,
        y_pos,
        f"  {label:.1%}",
        va="center",
        ha="left",
     )

ax.set_xlim(right=ax.get_xlim()[1] * 1.15)
save_figure(RESULTS_DIR / "figure-05.png")


In [ ]:
best_single = kld_plot.nsmallest(1, "KLD")
improvement = (best_single["KLD"] - res.fun) / best_single["KLD"]
print(f"Mixture improves over romance by {improvement.item():.1%}")


In [ ]:
p_mix = weighting @ Q
residuals = pd.DataFrame({
    "token": feature_names, "p_null": p_null, "p_mix": p_mix
})


In [ ]:
residuals["kl_contrib"] = rel_entr(residuals["p_null"], residuals["p_mix"])


In [ ]:
k = 25

under = (
    residuals
    .nlargest(k, "kl_contrib")
    .sort_values("kl_contrib", ascending=False)
    .assign(direction="Under-predicted")
)

over = (
    residuals
    .nsmallest(k, "kl_contrib")
    .sort_values("kl_contrib", ascending=False)
    .assign(direction="Over-predicted")
)

kl_plot = pd.concat([under, over], ignore_index=True)


In [ ]:
direction_order = ["Under-predicted", "Over-predicted"]

g = sns.relplot(
    data=kl_plot,
    x="kl_contrib",
    y="token",
    col="direction",
    col_order=direction_order,
    col_wrap=1,
    kind="scatter",
    hue="direction",
    hue_order=direction_order,
    facet_kws={"sharey": False, "sharex": True},
    height=FACET_HEIGHT,
    aspect=FACET_ASPECT,
    s=60,
    legend=False,
)
g.set_axis_labels("Signed token-level KL term", "")
g.set_titles("{col_name}")

limit = kl_plot["kl_contrib"].abs().max()
for ax in g.axes.flat:
    ax.axvline(0, color="0.3", linewidth=1)
    ax.set_xlim(-limit * 1.1, limit * 1.1)

g.fig.suptitle(
    "Tokens Under- and Over-predicted by the Fitted Mixture",
    y=1.05,
)
save_figure(RESULTS_DIR / "figure-06.png")


In [ ]:
top_plot.to_csv(RESULTS_DIR / "top_plot.csv", index=True)
kl_plot.to_csv(RESULTS_DIR / "kl_plot.csv", index=True)
mix_plot.to_csv(RESULTS_DIR / "mix_plot.csv", index=True)
residuals.to_csv(RESULTS_DIR / "residuals.csv", index=True)
observed = {name: module.__version__ for name, module in sys.modules.items()
            if name in {"numpy", "pandas", "torch", "transformers", "sklearn", "scipy", "matplotlib", "seaborn"}}
environment = {"python": platform.python_version(), "imported_packages": observed}
(RESULTS_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
print("Completed notebook; results:", RESULTS_DIR)
